# 🏖️ Indice di Overtourism — Sardegna
## Notebook 03 — Analisi della pressione turistica

**Obiettivo:** calcolare un indice composito di *overtourism* per i comuni della Sardegna,
scomponendolo in quattro pilastri tematici, normalizzandolo e visualizzandolo su mappa interattiva.

**Autore:** Analisi dati Sardegna  
**Versione:** 1.0


## 🔬 Obiettivo e distinzione concettuale

### Attrattività vs Pressione turistica

| Concetto | Descrizione | Dati chiave |
|---------|-------------|-------------|
| **Attrattività turistica** | Potenziale di un territorio di attrarre visitatori | Punti d'interesse naturalistici, musei, spiagge |
| **Pressione turistica (overtourism)** | Intensità dell'impatto dei flussi turistici sul tessuto locale | Densità ricettiva, saturazione commerciale, parcheggi |

> **Nota:** un territorio può essere *attrattivo* ma NON ancora in pressione (es. parchi naturali
> poco accessibili). Viceversa, un piccolo comune costiero con alta densità ricettiva può
> essere in forte pressione pur avendo poche attrazioni naturali.

### ⚠️ Avvertenze sui proxy OSM

I dati OpenStreetMap sono **volontari e variabili nel tempo**. I caveat principali:

1. **Copertura disomogenea** — zone rurali e periferiche sono spesso sottocartografate.
2. **Definizioni variabili** — `amenity=parking` include sia parcheggi pubblici che privati,
   parcheggi custoditi e slarghi informali.
3. **Nessun dato di flusso** — OSM non fornisce numero di turisti effettivi; i proxy sono
   proxy *areali* (densità per km²).
4. **Aggiornamento asincrono** — i tag `tourism=*` e `amenity=*` riflettono lo stato al
   momento del download, non dati real-time.


In [ ]:
# ============================================================
# CONFIGURAZIONE — modificare solo questa cella
# ============================================================
import os

# Percorso base dei dati (modificare se necessario)
DATA_DIR = os.environ.get("DATA_DIR", ".")

# File di input
OSM_PER_COMUNE  = os.path.join(DATA_DIR, "osm_per_comune.csv")
COMUNI_GEOJSON   = os.path.join(DATA_DIR, "comuni_sardegna.geojson")
OSM_RAW         = os.path.join(DATA_DIR, "osm_raw.json")

# File di output
OUTPUT_CSV      = os.path.join(DATA_DIR, "indice_overtourism.csv")
OUTPUT_HTML     = os.path.join(DATA_DIR, "mappa_overtourism_sardegna.html")

print(f"DATA_DIR   : {DATA_DIR}")
print(f"OSM csv    : {OSM_PER_COMUNE}")
print(f"GeoJSON    : {COMUNI_GEOJSON}")
print(f"OSM raw    : {OSM_RAW}")
print(f"Output CSV : {OUTPUT_CSV}")
print(f"Output HTML: {OUTPUT_HTML}")


In [ ]:
# ============================================================
# DIPENDENZE
# ============================================================
import subprocess, sys

pkgs = ["pandas","geopandas","shapely","folium","branca","matplotlib","seaborn"]
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])

import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
import branca.colormap as cm
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings("ignore")

print("OK: Tutte le dipendenze caricate.")


## 📂 Caricamento dati

Leggiamo i tre file di input:
- `osm_per_comune.csv` — aggregati OSM per comune (turismo, ristorazione, servizi, infrastrutture)
- `comuni_sardegna.geojson` — confini comunali
- `osm_raw.json` — dati OSM grezzi (per i parcheggi)


In [ ]:
# ============================================================
# CARICAMENTO DATI
# ============================================================

# --- osm_per_comune.csv ---
osm_df = pd.read_csv(OSM_PER_COMUNE)
# Normalizza codice ISTAT: 6 cifre con leading zeros
osm_df["com_istat_code"] = osm_df["com_istat_code"].apply(lambda x: f"{x:06d}")
print(f"[1] osm_per_comune.csv -> {len(osm_df)} comuni")
print(f"    Colonne: {list(osm_df.columns)}")

# --- comuni_sardegna.geojson ---
comuni_gdf = gpd.read_file(COMUNI_GEOJSON)
print(f"[2] comuni_sardegna.geojson -> {len(comuni_gdf)} comuni, CRS: {comuni_gdf.crs}")


## 📐 Calcolo dell'area comunale in km²

L'area è calcolata **riproiettando i confini in EPSG:32632 (UTM zona 32N)**,
un sistema di riferimento metrico adatto alla Sardegna. Il CRS originale (WGS84)
è solo per la memorizzazione; il calcolo geometrico richiede unità metriche.


In [ ]:
# ============================================================
# CALCOLO AREA COMUNALE IN KM2 (CRS metrico EPSG:32632)
# ============================================================
CRS_METRIC = 32632

print(f"CRS originale: {comuni_gdf.crs}")

# Riproietta in UTM 32N per calcolo area preciso
comuni_utm = comuni_gdf.to_crs(epsg=32632)

# Area in m2 -> km2
comuni_utm["area_m2"]  = comuni_utm.geometry.area
comuni_utm["area_km2"] = comuni_utm["area_m2"] / 1_000_000

# Riporta al gdf originale
comuni_gdf["area_km2"] = comuni_utm["area_km2"]

print(f"CRS riproiettato: {comuni_utm.crs}")
print(f"Area km2 -- min: {comuni_gdf['area_km2'].min():.4f}  max: {comuni_gdf['area_km2'].max():.4f}")
print(f"Comune piu grande: {comuni_gdf.loc[comuni_gdf['area_km2'].idxmax(), 'name']} "
      f"({comuni_gdf['area_km2'].max():.2f} km2)")


## 🅿️ Estrazione parcheggi da OSM raw

Per il pilastro 4 (*pressione parcheggi*) leggiamo `osm_raw.json` ed estraiamo
i nodi con `amenity=parking`. Ogni nodo viene associato a un comune tramite
**spatial join** (predicato `within`).

### ⚙️ Strategia di fallback

Se il file `osm_raw.json` non è disponibile o contiene **0 parcheggi utilizzabili**,
si attiva il **fallback documentato**:

```
pilastro4 = count_infrastrutture / area_km2
```

Il fallback utilizza `count_infrastrutture` (strade, stazioni, fermate, ponti)
come proxy proporzionale alla pressione infrastrutturale complessiva, normalizzata
per l'area. Questo non è identico a un conteggio parcheggi reale, ma è coerente
con la logica areale del resto dell'indice.


In [ ]:
# ============================================================
# ESTRAZIONE PARCHEGGI da osm_raw.json + spatial join
# ============================================================
import os
from shapely.geometry import Point

parking_counts = None
parking_nodes  = []
fallback_active = False

# --- tentativo di lettura osm_raw.json ---
if os.path.exists(OSM_RAW):
    try:
        with open(OSM_RAW, "r") as f:
            osm_data = json.load(f)
        elements = osm_data.get("elements", [])
        print(f"[osm_raw.json] -> {len(elements)} elementi totali")

        # Estrai nodi con amenity=parking
        for el in elements:
            if el.get("type") == "node":
                tags = el.get("tags", {})
                if tags.get("amenity") == "parking":
                    lat = el.get("lat")
                    lon = el.get("lon")
                    if lat and lon:
                        parking_nodes.append({"lat": lat, "lon": lon, "tags": tags})

        print(f"[osm_raw.json] -> {len(parking_nodes)} nodi parcheggio trovati")

        # Spatial join se ci sono parcheggi
        if len(parking_nodes) > 0:
            geometries = [Point(p["lon"], p["lat"]) for p in parking_nodes]
            parking_gdf = gpd.GeoDataFrame(geometry=geometries, crs="EPSG:4326")
            comuni_wgs84 = comuni_gdf.to_crs(epsg=4326)
            joined = gpd.sjoin(
                parking_gdf,
                comuni_wgs84[["com_istat_code", "geometry"]],
                how="left",
                predicate="within"
            )
            parking_counts = (
                joined.groupby("com_istat_code")
                .size()
                .reset_index(name="count_parcheggi")
            )
            print(f"[spatial join] -> {len(parking_counts)} comuni con parcheggi")
        else:
            print("[ATTENZIONE] Nessun parcheggio in osm_raw.json -- attivazione fallback.")

    except Exception as e:
        print(f"[ERRORE] Lettura osm_raw.json fallita: {e}")
        print("[ATTENZIONE] Attivazione fallback parcheggi.")
else:
    print(f"[ATTENZIONE] File osm_raw.json non trovato: {OSM_RAW}")
    print("[ATTENZIONE] Attivazione fallback parcheggi.")

# --- Fallback: count_infrastrutture / area_km2 ---
if parking_counts is None or len(parking_counts) == 0:
    fallback_active = True
    print("[FALLBACK ATTIVO] Pilastro 4 = count_infrastrutture / area_km2")
    parking_counts = None   # il pilastro 4 userà il fallback


## 🔗 Merge dei dati

Uniamo i dati OSM aggregati ai confini comunali, selezionando solo le colonne
necessarie per evitare conflitti.


In [ ]:
# ============================================================
# MERGE DATI
# ============================================================
comuni_subset = comuni_gdf[["com_istat_code", "name", "area_km2"]].copy()
comuni_subset = comuni_subset.rename(columns={"name": "nome_comune"})

df = comuni_subset.merge(osm_df, on="com_istat_code", how="inner")
print(f"Comuni con dati completi: {len(df)}")
print(f"Colonne disponibili: {list(df.columns)}")
print(df[["nome_comune","com_istat_code","area_km2","count_turismo",
         "count_ristorazione","count_servizi","count_infrastrutture"]].head())


## 🧮 Calcolo dei quattro pilastri

| Pilastro | Indicatore | Peso | Formula |
|----------|-----------|------|---------|
| **P1** | Densita ricettiva | 35% | `count_turismo / area_km2` |
| **P2** | Saturazione commerciale turistica | 25% | `count_ristorazione / area_km2` |
| **P3** | Squilibrio turismo/servizi | 25% | `count_turismo / (count_servizi + 1)` |
| **P4** | Pressione parcheggi | 15% | `count_parcheggi / area_km2` *(o fallback)* |

**Nota su P3:** il `+1` al denominatore evita la divisione per zero nei comuni
senza servizi. Il rapporto esprime quanto il turismo supera l'offerta di servizi locali.


In [ ]:
# ============================================================
# CALCOLO PILASTRI GREZZI
# ============================================================

# Pilastro 1: densita ricettiva (35%)
df["p1_raw"] = df["count_turismo"] / df["area_km2"]

# Pilastro 2: saturazione commerciale (25%)
df["p2_raw"] = df["count_ristorazione"] / df["area_km2"]

# Pilastro 3: squilibrio turismo/servizi (25%)
df["p3_raw"] = df["count_turismo"] / (df["count_servizi"] + 1)

# Pilastro 4: pressione parcheggi (15%)
if parking_counts is not None and len(parking_counts) > 0:
    df = df.merge(parking_counts, on="com_istat_code", how="left")
    df["count_parcheggi"] = df["count_parcheggi"].fillna(0)
    df["p4_raw"] = df["count_parcheggi"] / df["area_km2"]
    p4_source = "count_parcheggi / area_km2"
else:
    # Fallback documentato
    df["p4_raw"] = df["count_infrastrutture"] / df["area_km2"]
    p4_source = "count_infrastrutture / area_km2  [FALLBACK]"

print(f"Pilastro 4 calcolato con: {p4_source}")
print()
print("Statistiche pilastri grezzi:")
for p in ["p1_raw","p2_raw","p3_raw","p4_raw"]:
    print(f"  {p}: min={df[p].min():.4f}  max={df[p].max():.4f}  mean={df[p].mean():.4f}")


## ⚖️ Normalizzazione Min-Max (0-100)

Ogni pilastro è normalizzato con la formula:

    score_i = (x_i - min(x)) / (max(x) - min(x)) * 100

Se il valore minimo coincide con il massimo (caso degenere), si assegna il valore **50.0**
per tutti i comuni, evitando la divisione per zero.


In [ ]:
# ============================================================
# NORMALIZZAZIONE MIN-MAX 0-100
# ============================================================
def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([50.0] * len(series), index=series.index)
    return (series - min_val) / (max_val - min_val) * 100

df["score_densita_ricettiva"]       = min_max_normalize(df["p1_raw"])
df["score_saturazione_commerciale"] = min_max_normalize(df["p2_raw"])
df["score_squilibrio"]              = min_max_normalize(df["p3_raw"])
df["score_pressione_parcheggi"]     = min_max_normalize(df["p4_raw"])

print("Normalizzazione completata.")
for col in ["score_densita_ricettiva","score_saturazione_commerciale",
            "score_squilibrio","score_pressione_parcheggi"]:
    print(f"  {col}: min={df[col].min():.2f}  max={df[col].max():.2f}")


## 🏆 Calcolo dell'indice composito

L'indice di overtourism è una **media ponderata** dei quattro pilastri normalizzati:

    Indice = 0.35 * P1 + 0.25 * P2 + 0.25 * P3 + 0.15 * P4

I pesi riflettono l'importanza relativa:
- **35%** — la densita ricettiva è il segnale più diretto di pressione
- **25%** — la saturazione commerciale indica un tessuto economico distorto dal turismo
- **25%** — lo squilibrio turismo/servizi segnala carenza strutturale
- **15%** — la pressione parcheggi misura l'impatto infrastrutturale


In [ ]:
# ============================================================
# INDICE COMPOSITO  (0.35 + 0.25 + 0.25 + 0.15 = 1.00)
# ============================================================
PESI = {"densita_ricettiva": 0.35,
        "saturazione_commerciale": 0.25,
        "squilibrio": 0.25,
        "pressione_parcheggi": 0.15}

print("Pesi applicati:")
for k, v in PESI.items():
    print(f"  {k}: {v:.0%}")
print(f"  Totale: {sum(PESI.values()):.0%}")

df["indice_overtourism"] = (
    PESI["densita_ricettiva"]       * df["score_densita_ricettiva"] +
    PESI["saturazione_commerciale"] * df["score_saturazione_commerciale"] +
    PESI["squilibrio"]              * df["score_squilibrio"] +
    PESI["pressione_parcheggi"]     * df["score_pressione_parcheggi"]
)

print()
print(f"Indice overtourism: min={df['indice_overtourism'].min():.2f}  "
      f"max={df['indice_overtourism'].max():.2f}  "
      f"media={df['indice_overtourism'].mean():.2f}  "
      f"mediana={df['indice_overtourism'].median():.2f}")


## 📊 Classificazione in quintili (1-5)

L'indice composito è suddiviso in **5 classi** (quintili) per facilitare
la lettura e la comunicazione:

| Classe | Significato |
|--------|-------------|
| 1 | Overtourism molto basso (quintile inferiore) |
| 2 | Overtourism basso |
| 3 | Overtourism moderato |
| 4 | Overtourism alto |
| 5 | Overtourism molto alto (quintile superiore)


In [ ]:
# ============================================================
# CLASSIFICAZIONE IN QUINTILI 1-5
# ============================================================
df["classe"] = pd.qcut(
    df["indice_overtourism"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

print("Distribuzione classi:")
print(df["classe"].value_counts().sort_index())
print()
print("Cutoff quintili:")
cuts = pd.qcut(df["indice_overtourism"], q=5, retbins=True)[1]
for i in range(len(cuts)-1):
    print(f"  Classe {i+1}: {cuts[i]:.2f} - {cuts[i+1]:.2f}")


## ✅ Controlli di qualita

Verifichiamo:
- Assenza di NaN nei pilastri e nell'indice
- Valori entro il range atteso [0, 100]
- Copertura dei comuni


In [ ]:
# ============================================================
# CONTROLLO QUALITA
# ============================================================
score_cols = ["score_densita_ricettiva","score_saturazione_commerciale",
               "score_squilibrio","score_pressione_parcheggi","indice_overtourism"]

print("=== Controllo NaN ===")
for col in score_cols + ["classe"]:
    n_nan = df[col].isna().sum()
    print(f"  {col}: NaN = {n_nan}")

print()
print("=== Controllo range [0, 100] ===")
for col in score_cols:
    lo, hi = df[col].min(), df[col].max()
    ok = "OK" if 0 <= lo and hi <= 101 else "FAIL"
    print(f"  [{ok}] {col}: [{lo:.2f}, {hi:.2f}]")

print()
print("=== Copertura ===")
print(f"  Comuni in output: {len(df)}")
print(f"  Comuni totali in GeoJSON: {len(comuni_gdf)}")
print(f"  Comuni in OSM csv: {len(osm_df)}")
print(f"  NaN totali nel dataframe: {df[score_cols].isna().sum().sum()}")


## 💾 Esportazione CSV

Esportiamo il dataframe completo con l'indice e le classi in
`indice_overtourism.csv`.


In [ ]:
# ============================================================
# ESPORTAZIONE CSV
# ============================================================
output_df = df[[
    "nome_comune", "com_istat_code", "area_km2",
    "score_densita_ricettiva", "score_saturazione_commerciale",
    "score_squilibrio", "score_pressione_parcheggi",
    "indice_overtourism", "classe"
]].copy()

output_df.columns = [
    "nome_comune", "codice_istat", "area_km2",
    "score_densita_ricettiva", "score_saturazione_commerciale",
    "score_squilibrio", "score_pressione_parcheggi",
    "indice_overtourism", "classe"
]

# Arrotonda
for col in ["area_km2","score_densita_ricettiva","score_saturazione_commerciale",
            "score_squilibrio","score_pressione_parcheggi","indice_overtourism"]:
    output_df[col] = output_df[col].round(2)

# Ordina per indice overtourism decrescente
output_df = output_df.sort_values("indice_overtourism", ascending=False).reset_index(drop=True)

output_df.to_csv(OUTPUT_CSV, index=False)
print(f"OK: File CSV salvato: {OUTPUT_CSV}")
print(f"  Righe: {len(output_df)}")
print()
print(output_df.head(10).to_string(index=False))


## 🏅 Top 10 - Comuni a maggiore rischio overtourism


In [ ]:
# ============================================================
# TOP 10 COMUNI A RISCHIO
# ============================================================
top10 = output_df.head(10).copy()
top10.index = range(1, 11)

print("TOP 10 -- Comuni a maggiore rischio overtourism")
print("=" * 65)
print(f"{'#':>3}  {'Comune':<28} {'ISTAT':<8} {'Indice':>7} {'Classe':>6}")
print("-" * 65)
for i, (_, row) in enumerate(top10.iterrows(), 1):
    print(f"{i:>3}  {row['nome_comune']:<28} {row['codice_istat']:<8} "
          f"{row['indice_overtourism']:>7.2f} {row['classe']:>6}")
print("-" * 65)

# Mostra pilastri
print()
print("Dettaglio pilastri per i top 10:")
print(top10[["nome_comune","score_densita_ricettiva","score_saturazione_commerciale",
            "score_squilibrio","score_pressione_parcheggi","indice_overtourism"]].to_string(index=False))


## 📈 Visualizzazione comparativa dei quattro pilastri

Grafico a barre raggruppate per i **top 10 comuni**, mostrando il contributo
di ciascun pilastro all'indice composito.


In [ ]:
# ============================================================
# VISUALIZZAZIONE COMPARATIVA DEI 4 PILASTRI (TOP 10)
# ============================================================
PILASTRI = {
    "score_densita_ricettiva":       ("P1 Densita ricettiva",        "#2E86AB"),
    "score_saturazione_commerciale": ("P2 Saturazione commerciale",  "#A23B72"),
    "score_squilibrio":              ("P3 Squilibrio turismo/serv.", "#F18F01"),
    "score_pressione_parcheggi":     ("P4 Pressione parcheggi",      "#C73E1D"),
}

top10_viz = output_df.head(10).copy()
top10_viz = top10_viz.sort_values("indice_overtourism", ascending=True)

fig, ax = plt.subplots(figsize=(13, 7))
x = range(len(top10_viz))
bar_width = 0.18
offset = -(len(PILASTRI)-1)/2

for i, (col, (label, color)) in enumerate(PILASTRI.items()):
    vals = top10_viz[col].values
    ax.bar([p + (offset + i)*bar_width for p in x], vals,
           bar_width, label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(top10_viz["nome_comune"].values, rotation=35, ha="right", fontsize=9)
ax.set_ylabel("Score normalizzato (0-100)", fontsize=11)
ax.set_title("Top 10 Comuni -- Confronto Pilastri Overtourism (Sardegna)", fontsize=13, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
ax.set_ylim(0, 110)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(DATA_DIR, "comparativa_pilastri_top10.png"), dpi=150)
plt.show()
print("OK: Grafico salvato: comparativa_pilastri_top10.png")


## 🗺️ Mappa interattiva Folium

Mappa coropletica della Sardegna con palette **verde -> rosso**:
- Verde chiaro -> classe 1 (basso rischio)
- Rosso scuro -> classe 5 (alto rischio)

Elementi inclusi:
- **Tooltip** con nome comune, indice e pilastri
- **Legenda** colorimetrica con classi
- **Pannello metodologico** con descrizione della formula


In [ ]:
# ============================================================
# MAPPA FOLIUM -- COROPLETICA OVERTURISM
# ============================================================
import branca.colormap as cm
import folium

# Prepara GeoDataFrame per la mappa
map_gdf = comuni_gdf.merge(
    output_df[["codice_istat","nome_comune","area_km2",
               "score_densita_ricettiva","score_saturazione_commerciale",
               "score_squilibrio","score_pressione_parcheggi",
               "indice_overtourism","classe"]],
    left_on="com_istat_code",
    right_on="codice_istat",
    how="inner"
)

# Centro mappa = bounding box della Sardegna
bounds = map_gdf.total_bounds
center_lat = (bounds[1] + bounds[3]) / 2
center_lon = (bounds[0] + bounds[2]) / 2

m = folium.Map(location=[center_lat, center_lon], zoom_start=7,
               tiles="CartoDB positron", attr="CartoDB")

# Colormap verde -> rosso
colormap = cm.LinearColormap(
    colors=["#1a9850","#91cf60","#d9ef8b","#fee08b","#d73027"],
    vmin=1, vmax=5,
    caption="Classe Overtourism (1=basso, 5=molto alto)"
)

def style_fn(feature):
    cls = feature["properties"].get("classe", 3)
    return {
        "fillColor": colormap(cls),
        "color": "#444444",
        "weight": 0.5,
        "fillOpacity": 0.75
    }

def highlight_fn(feature):
    return {
        "fillColor": colormap(feature["properties"].get("classe", 3)),
        "color": "#000000",
        "weight": 2,
        "fillOpacity": 0.9
    }

tooltip = folium.GeoJsonTooltip(
    fields=["nome_comune","classe","indice_overtourism",
            "score_densita_ricettiva","score_saturazione_commerciale",
            "score_squilibrio","score_pressione_parcheggi"],
    aliases=["Comune","Classe (1-5)","Indice Overtourism",
             "P1 Densita ricettiva","P2 Saturazione comm.","P3 Squilibrio","P4 Parcheggi"],
    localize=True,
    sticky=True
)

folium.GeoJson(
    map_gdf,
    style_function=style_fn,
    highlight_function=highlight_fn,
    tooltip=tooltip,
    name="Overtourism Sardegna"
).add_to(m)

colormap.add_to(m)

# Pannello metodologico HTML
method_html = """<div style="position:fixed;bottom:20px;left:20px;z-index:9999;
background-color:white;padding:14px;border-radius:8px;
border:1px solid #ccc;max-width:320px;font-size:12px;
box-shadow:0 2px 6px rgba(0,0,0,0.3);">
<b>Metodologia -- Indice Overtourism</b><br><br>
<b>Formula:</b><br>
Indice = 0.35*P1 + 0.25*P2 + 0.25*P3 + 0.15*P4<br><br>
<b>Pilastri:</b><br>
P1: Densita ricettiva (turismo/km2)<br>
P2: Saturazione commerciale (ristoraz./km2)<br>
P3: Squilibrio (turismo/servizi+1)<br>
P4: Parcheggi (o fallback infrastrutture/km2)<br><br>
<b>Normalizzazione:</b> min-max -> 0-100<br>
<b>Classi:</b> quintili 1-5<br><br>
<b>Proxy OSM:</b> dati volontari, copertura variabile.
</div>"""

m.get_root().html.add_child(folium.Element(method_html))

m.save(OUTPUT_HTML)
print(f"OK: Mappa Folium salvata: {OUTPUT_HTML}")
m


## 📝 Riepilogo

Questo notebook ha calcolato l'**indice di overtourism** per i comuni della Sardegna:

1. Caricato i dati OSM aggregati per comune
2. Calcolato l'area comunale in km2 (CRS metrico EPSG:32632)
3. Estratto i parcheggi da `osm_raw.json` con spatial join
4. Applicato **fallback parcheggi** (count_infrastrutture) se OSM raw non utilizzabile
5. Calcolato i 4 pilastri normalizzati (min-max 0-100)
6. Composizione ponderata -> indice 0-100
7. Classificazione in quintili 1-5
8. Controlli di qualita (NaN, range, copertura)
9. Esportato `indice_overtourism.csv`
10. Generato `mappa_overtourism_sardegna.html` interattiva
11. Visualizzazione comparativa dei pilastri
